## Course data paths

Run the setup cell below before the lesson. Bundled inputs are resolved from the course repository, so paths also work after moving the notebook into `code/`. Generated practice files still use the notebook working directory.


In [ ]:
from pathlib import Path

# Find the cloned course repository from the notebook or repository folder.
COURSE_ROOT = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "docs" / "course-catalog.json").is_file()),
    None,
)
if COURSE_ROOT is None:
    raise FileNotFoundError("Open this notebook inside the cloned course repository.")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

# Load dataset
data = pd.read_csv(COURSE_ROOT / "modules/06-machine-learning/datasets/breast_cancer.csv")

# Drop unnecessary columns
data = data.drop(['id'], axis=1)

# Encode target
data['diagnosis'] = data['diagnosis'].map({'M': 1, 'B': 0})

# Select two features
X = data[['radius_mean', 'texture_mean']].values
y = data['diagnosis'].values

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Normalize
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# Linear Kernel
model_linear = SVC(kernel='linear')
model_linear.fit(X_train, y_train)

# Polynomial Kernel (degree = 3)
model_poly = SVC(kernel='poly', degree=3)
model_poly.fit(X_train, y_train)

# Accuracy
print(f"Linear Kernel Accuracy : {accuracy_score(y_test, model_linear.predict(X_test)) * 100:.2f}%")
print(f"Poly   Kernel Accuracy : {accuracy_score(y_test, model_poly.predict(X_test))   * 100:.2f}%")

def plot_decision_boundary(model, X, y, title, ax):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))

    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')

    ax.scatter(X[y == 0, 0], X[y == 0, 1],
               label='Benign', color='blue', edgecolors='k', s=40)

    ax.scatter(X[y == 1, 0], X[y == 1, 1],
               label='Malignant', color='red', edgecolors='k', s=40)

    ax.set_title(title)
    ax.set_xlabel('radius_mean (normalized)')
    ax.set_ylabel('texture_mean (normalized)')
    ax.legend()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_decision_boundary(model_linear, X_test, y_test,
                       'SVM — Linear Kernel', axes[0])

plot_decision_boundary(model_poly, X_test, y_test,
                       'SVM — Polynomial Kernel', axes[1])

plt.tight_layout()
plt.show()



In [ ]:
def plot_cm(model, X, y, title, ax):
    cm = confusion_matrix(y, model.predict(X))

    ax.imshow(cm, cmap='Blues')
    ax.set_title(title)

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])

    ax.set_xticklabels(['Pred B', 'Pred M'])
    ax.set_yticklabels(['True B', 'True M'])

    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j],
                    ha='center', va='center', fontsize=14)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

plot_cm(model_linear, X_test, y_test,
        'Confusion Matrix — Linear', axes[0])

plot_cm(model_poly, X_test, y_test,
        'Confusion Matrix — Poly', axes[1])

plt.tight_layout()
plt.show()

In [ ]:
model_poly5 = SVC(kernel='poly', degree=5)
model_poly5.fit(X_train, y_train)

print("Degree 5 Accuracy:",
      accuracy_score(y_test, model_poly5.predict(X_test)))

model_poly10 = SVC(kernel='poly', degree=10)
model_poly10.fit(X_train, y_train)

print("Degree 10 Accuracy:",
      accuracy_score(y_test, model_poly10.predict(X_test)))

### Q1. Which kernel gave higher accuracy? Why?

In most cases, the Linear kernel performs slightly better or similar on this dataset.

Reason:

The selected features (radius_mean, texture_mean) are almost linearly separable
Linear SVM finds a clean margin without overfitting
Polynomial kernel may introduce unnecessary complexity
Q2. Decision Boundary Difference
Linear Kernel:
Straight line boundary
Simple and interpretable
Polynomial Kernel:
Curved boundary
Can capture non-linear relationships

👉 In your plot:

Linear → straight separation
Polynomial → curved/complex regions
Q3. Confusion Matrix Analysis

Confusion matrix shows:

False Positive (FP):
Predict cancer when it's not
False Negative (FN):
Predict no cancer when it actually exists
⚠️ Most Dangerous:

👉 False Negative

Why?

Cancer is present but not detected
Patient may miss early treatment
Can lead to severe consequences or death